<!-- KERNEL_BANNER -->
> **Use kernel: `mrigi_tor190_v8`**
>
> Set the notebook kernel to *Python (mrigi_tor190_v8)* before running.

# 30d: Aggregate an existing single-judge `judge_results_final_*.json` (no re-judging)

Consumes an existing GPT-4.1 judge result file produced by **notebook 30** (single judge, single model, k × category × method × question).

Runs the **same viz layout as 30b** starting from its `## Aggregate to a summary DataFrame + CSV` section, adapted to this file's axes:

| 30b cell | 30b axes | 30d equivalent |
|---|---|---|
| 14 aggregate | rows per (model, method, judge) | rows per (k, category, method) |
| 16 bars per task | x = models, grouped by (method × judge) | x = methods, grouped by (category × k) |
| 17 heatmap | one per (judge, method); rows = models, cols = tasks | one per (k, category); rows = methods, cols = tasks |
| 19-20 agreement / scatter | gpt-4.1 vs gpt-4o | **omitted** — single-judge file, nothing to compare |
| 21 final printout | dual-judge metadata | single-judge metadata |

No re-judging is performed. If the file has fewer than the four tasks (relevance / correctness / completeness / faithfulness), the missing ones are skipped gracefully — this old file has three (no faithfulness).

In [ ]:
import os
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# The file the user pointed at. Set to None to auto-pick the newest
# `judge_results_final_*.json` in this directory.
JUDGE_FILE = 'judge_results_final_20260306_162420.json'

if JUDGE_FILE is None or not os.path.exists(JUDGE_FILE):
    candidates = sorted(glob.glob('judge_results_final_*.json'))
    if not candidates:
        raise FileNotFoundError("No judge_results_final_*.json in current directory.")
    JUDGE_FILE = candidates[-1]

with open(JUDGE_FILE) as f:
    payload = json.load(f)

META = payload.get('metadata', {})
RES  = payload.get('results', payload)  # fall back if there's no metadata wrapper

MODEL_EVALUATED = META.get('model_evaluated', '(unknown)')
JUDGE_MODEL     = META.get('judge_model', '(unknown)')
K_LABELS        = META.get('k_labels', sorted(RES.keys()))
CATEGORIES      = META.get('categories', sorted({c for k in RES for c in RES[k]}))
METHODS         = META.get('methods',   sorted({m for k in RES for c in RES[k] for m in RES[k][c]}))
TIMESTAMP_SRC   = META.get('timestamp', 'unknown')

print(f"Loaded:            {JUDGE_FILE}  ({os.path.getsize(JUDGE_FILE)/1e6:.1f} MB)")
print(f"Model evaluated:   {MODEL_EVALUATED}")
print(f"Judge:             {JUDGE_MODEL}")
print(f"Source timestamp:  {TIMESTAMP_SRC}")
print(f"K labels:          {K_LABELS}")
print(f"Categories:        {CATEGORIES}")
print(f"Methods:           {METHODS}")
print(f"Total questions:   {META.get('total_questions_judged', '(unknown)')}")

## Aggregate to a summary DataFrame + CSV

One row per `(k_label, category, method)` with mean and n for each task present in the source file.

In [ ]:
# Discover which tasks are present by scanning the first non-empty record.
def _sniff_tasks(res):
    for k in res:
        for c in res[k]:
            for m in res[k][c]:
                for r in res[k][c][m]:
                    keys = list(r.keys())
                    return [k[:-len('_score')] for k in keys if k.endswith('_score')]
    return []

TASKS = _sniff_tasks(RES)
print(f"Tasks discovered in file: {TASKS}")

rows = []
for k_label in K_LABELS:
    if k_label not in RES:
        continue
    for cat in CATEGORIES:
        if cat not in RES[k_label]:
            continue
        for method in METHODS:
            records = RES[k_label][cat].get(method, [])
            row = {
                'k': k_label, 'category': cat, 'method': method,
                'n_questions': len(records),
            }
            for task in TASKS:
                scores = [r.get(f'{task}_score', -1) for r in records]
                valid  = [s for s in scores if isinstance(s, int) and s > 0]
                row[f'{task}_mean'] = float(np.mean(valid)) if valid else np.nan
                row[f'{task}_std']  = float(np.std(valid))  if valid else np.nan
                row[f'{task}_n']    = len(valid)
            rows.append(row)

summary_df = pd.DataFrame(rows)

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
summary_csv = f'judge_summary_30d_{ts}.csv'
summary_df.to_csv(summary_csv, index=False)
print(f"✓ Summary CSV: {summary_csv}  ({len(summary_df)} rows)")
summary_df

In [ ]:
# Overall means per method (averaged across categories) at each k — quick sanity view.
print("Overall means per method (averaged across categories):")
for k_label in K_LABELS:
    sub = summary_df[summary_df['k'] == k_label]
    if sub.empty:
        continue
    print(f"\n  [{k_label}]")
    header = f"    {'method':<20}" + ''.join(f"  {task:>14}" for task in TASKS)
    print(header)
    print('    ' + '-' * (len(header) - 4))
    for method in METHODS:
        row_line = f"    {method:<20}"
        m_sub = sub[sub['method'] == method]
        for task in TASKS:
            col = f'{task}_mean'
            val = np.nanmean(m_sub[col]) if col in m_sub.columns else np.nan
            row_line += f"  {(f'{val:.2f}' if not np.isnan(val) else 'N/A'):>14}"
        print(row_line)

## Visualizations

For each `(task, k)` combo: one bar chart with methods on the x-axis and categories as grouped bars, plus one heatmap with methods × categories.

In [ ]:
# Plot: per-task bar chart with (CATEGORY × K) grouped bars per method.
# Mirrors 30b cell 16 — same structure, adapted to the axes present in this file
# (single model, single judge; primary comparison is 7 methods; secondary dims are k and category).
TASK_TITLES = {
    'relevance':    'Context Relevance',
    'correctness':  'Response Correctness',
    'completeness': 'Response Completeness',
    'faithfulness': 'Response Faithfulness',
}
CATEGORY_COLORS = {
    'Synthesis': '#1f77b4',
    'ChemCon':   '#ff7f0e',
    'DAC':       '#2ca02c',
}
_fallback_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
K_HATCH = {K_LABELS[0]: '', K_LABELS[1]: '///'} if len(K_LABELS) >= 2 else {K_LABELS[0]: ''}

for task in TASKS:
    task_col   = f'{task}_mean'
    task_title = TASK_TITLES.get(task, task.capitalize())

    fig, ax = plt.subplots(figsize=(max(14, len(METHODS) * 1.6), 6))
    x = np.arange(len(METHODS))

    combos = [(cat, k) for cat in CATEGORIES for k in K_LABELS]  # 3 × 2 = 6 combos
    width  = 0.8 / max(1, len(combos))

    for i, (cat, k_label) in enumerate(combos):
        sub = (summary_df[(summary_df['category'] == cat) & (summary_df['k'] == k_label)]
               .set_index('method').reindex(METHODS))
        vals = sub[task_col].fillna(0).values
        offset = (i - (len(combos) - 1) / 2) * width
        color  = CATEGORY_COLORS.get(cat, _fallback_cycle[list(CATEGORIES).index(cat) % len(_fallback_cycle)])
        bars = ax.bar(x + offset, vals, width,
                      label=f'{cat} / {k_label}',
                      color=color, alpha=0.85, edgecolor='black',
                      hatch=K_HATCH.get(k_label, ''))
        for bar, v in zip(bars, vals):
            if not np.isnan(v) and v > 0:
                ax.text(bar.get_x() + bar.get_width()/2., v + 0.1, f'{v:.1f}',
                        ha='center', va='bottom', fontsize=6, fontweight='bold')

    ax.set_xlabel('Method', fontsize=11, fontweight='bold')
    ax.set_ylabel(f'{task_title} (mean, 1-10)', fontsize=11, fontweight='bold')
    ax.set_title(f'30d — {task_title}: category × k  ·  {MODEL_EVALUATED} · judge={JUDGE_MODEL}',
                 fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(METHODS, rotation=45, ha='right', fontsize=9)
    ax.set_ylim(0, 10.5)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.legend(fontsize=9, ncol=len(CATEGORIES))
    plt.tight_layout()
    out = f'judge_30d_bars_{task_col}_{ts}.svg'
    fig.savefig(out, format='svg', bbox_inches='tight')
    plt.show()
    print(f'✓ Saved: {out}')

In [ ]:
# Heatmap: methods × tasks, one heatmap per (k, category).
# Mirrors 30b cell 17 — same structure, adapted so the "primary comparison" axis
# (methods here, models in 30b) becomes the rows and TASKS are the columns.
task_cols   = [f'{t}_mean' for t in TASKS]
task_labels = [TASK_TITLES.get(t, t.capitalize()) for t in TASKS]

for k_label in K_LABELS:
    for cat in CATEGORIES:
        sub = (summary_df[(summary_df['k'] == k_label) & (summary_df['category'] == cat)]
               .set_index('method').reindex(METHODS))
        data = sub[task_cols].values.astype(float)

        fig, ax = plt.subplots(figsize=(max(7, len(TASKS) * 2), max(5, len(METHODS) * 0.55)))
        im = ax.imshow(data, aspect='auto', cmap='YlGnBu', vmin=0, vmax=10)

        ax.set_xticks(range(len(task_labels)))
        ax.set_xticklabels(task_labels, fontsize=10)
        ax.set_yticks(range(len(METHODS)))
        ax.set_yticklabels(METHODS, fontsize=9)
        ax.set_title(f'30d — {JUDGE_MODEL} · {cat} · {k_label}', fontsize=12, fontweight='bold')

        for i in range(data.shape[0]):
            for j in range(data.shape[1]):
                val = data[i, j]
                txt = f'{val:.1f}' if not np.isnan(val) else 'N/A'
                color = 'white' if (not np.isnan(val) and val > 6.5) else 'black'
                ax.text(j, i, txt, ha='center', va='center', fontsize=9, color=color, fontweight='bold')

        fig.colorbar(im, ax=ax, label='mean score (1-10)')
        plt.tight_layout()
        out = f'judge_30d_heatmap_{cat}_{k_label.replace("=", "")}_{ts}.svg'
        fig.savefig(out, format='svg', bbox_inches='tight')
        plt.show()
        print(f'✓ Saved: {out}')

In [ ]:
# Final summary printout (mirrors 30b cell 21 structure)
print('=' * 80)
print('Mrigi 30d — Aggregation of existing single-judge JSON complete')
print('=' * 80)
print(f'Source judge JSON:  {JUDGE_FILE}')
print(f'Model evaluated:    {MODEL_EVALUATED}')
print(f'Judge:              {JUDGE_MODEL}')
print(f'K labels:           {K_LABELS}')
print(f'Categories:         {CATEGORIES}')
print(f'Methods:            {METHODS}')
print(f'Tasks:              {TASKS}')
print(f'Summary CSV:        {summary_csv}')
print(f'Bar charts (SVG):   judge_30d_bars_*_{ts}.svg  ({len(TASKS)} figures — one per task)')
print(f'Heatmaps (SVG):     judge_30d_heatmap_*_{ts}.svg  ({len(K_LABELS) * len(CATEGORIES)} figures — one per (k, category))')
print('=' * 80)
print()
print('(Note: judge-agreement + gpt-4.1-vs-gpt-4o scatter from 30b cells 19-20 are')
print(' intentionally omitted — this input is single-judge, nothing to compare against.)')